<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_5_evaluar_representaciones.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/04_representation/notebooks/sesion_04_5_evaluar_representaciones.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F04_representation%2Fnotebooks%2Fsesion_04_5_evaluar_representaciones.ipynb)

# SI7011 · Sesión 04 · 5. Evaluar representaciones
**Juan David Martínez Vargas · Universidad EAFIT**

Acompaña la **Parte 5** de las diapositivas. Tenemos cuatro encoders con la **misma arquitectura** (ResNet-18) y cuatro objetivos distintos. Aquí los medimos igual, con las tres lecturas de la diapositiva *Three ways to read a representation*: $k$-NN, *linear probe* y *fine-tuning*.

| Encoder $g$ | Cómo se entrenó | Etiquetas usadas |
|---|---|---|
| aleatorio | no se entrenó | ninguna |
| autoencoder | reconstruir CIFAR-10 (sección 0) | ninguna |
| SimCLR | InfoNCE sobre CIFAR-10 (notebook 4) | ninguna |
| ImageNet | clasificación supervisada | 1.2 millones, de otro dataset |

| Sección | Lo que medimos | Qué mirar |
|---|---|---|
| 1 | $k$-NN | sin entrenar nada |
| 2 | recuperación de imágenes | los vecinos, y sus errores |
| 3 | *linear probe* con todas las etiquetas | el techo de cada $\mathbf h$ |
| 4 | *probe* con 1, 10, 100, 1000 y 5000 etiquetas por clase | la figura de la diapositiva |
| 5 | t-SNE | mirar no es medir |
| 6 | 10 etiquetas por clase: $k$-NN, *probe* o *fine-tuning* | la pregunta de la diapositiva |

**Usa GPU**: unos 15 minutos. **En Kaggle:** agrega con **Add Input** el dataset [`pankrzysiu/cifar10-python`](https://www.kaggle.com/datasets/pankrzysiu/cifar10-python) y el modelo [`juanmartinezv4399/si7011-s04-simclr`](https://www.kaggle.com/models/juanmartinezv4399/si7011-s04-simclr) (pestaña **Models**), que trae el encoder SimCLR (o usa el `simclr_g.pt` que guardó el notebook 4).

In [ ]:
import copy, json, math, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import datasets
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
amp = device.type == 'cuda'
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
print('PyTorch', torch.__version__, '· device', device)

### Cargar CIFAR-10
La celda busca los datos en este orden y usa el primero que encuentre. Imprime la **fuente** usada: revísala.

| Plataforma | De dónde salen los datos |
|---|---|
| **Kaggle** | el dataset [`pankrzysiu/cifar10-python`](https://www.kaggle.com/datasets/pankrzysiu/cifar10-python) agregado con **Add Input** (`/kaggle/input/...`), sin descargar |
| **Lightning Studio** o **local** | la carpeta `data/` junto al notebook, si ya tiene los archivos de una ejecución anterior |
| **Colab** | `kagglehub` con el mismo dataset y tu secreto `KAGGLE_API_TOKEN` (el mismo de la sesión 01) |
| sin dataset ni token | descarga del sitio original con `torchvision` (unos 170 MB; puede tardar varios minutos) |

Es el mismo cargador de la sesión 03: cinco lotes de entrenamiento y uno de prueba, cada uno con 10 000 imágenes guardadas como filas de 3 072 bytes (R, luego G, luego B).

In [ ]:
import os, pickle, tarfile
from pathlib import Path

KAGGLE_HANDLE = 'pankrzysiu/cifar10-python'          # CIFAR-10 en el formato original para Python
ROOTS = [Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def kaggle_token():
    # Colab: secreto KAGGLE_API_TOKEN. Local o Lightning: variable de entorno o ~/.kaggle/access_token.
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

def find_batches(roots):
    for root in [r for r in roots if r.exists()]:
        hit = next(root.rglob('data_batch_1'), None)
        if hit is None:                                    # algunas copias traen solo el comprimido
            tgz = next(root.rglob('cifar-10-python.tar.gz'), None)
            if tgz is not None:
                print('Descomprimiendo', tgz, '→ data/')
                with tarfile.open(tgz) as t:
                    t.extractall('data')
                hit = next(Path('data').rglob('data_batch_1'), None)
        if hit is not None:
            return hit.parent
    return None

def locate_cifar10():
    folder = find_batches(ROOTS)
    if folder is not None:
        return folder, 'input de Kaggle' if str(folder).startswith('/kaggle/input') else 'archivos locales'
    if kaggle_token() or Path('/kaggle/input').exists():
        try:
            import kagglehub                               # preinstalado en Colab y Kaggle
            folder = find_batches([Path(kagglehub.dataset_download(KAGGLE_HANDLE))])
            if folder is not None:
                return folder, 'kagglehub'
        except Exception as e:
            print('kagglehub no disponible:', type(e).__name__, '→ se descarga del sitio original')
    datasets.CIFAR10('data', download=True)                # el mismo formato, en data/cifar-10-batches-py
    return find_batches([Path('data')]), 'torchvision (sitio original)'

def read_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    # cada fila: 3 072 bytes = R (1 024) | G (1 024) | B (1 024) → 3 × 32 × 32 → H × W × C, como un PNG
    x = np.asarray(d[b'data'], dtype=np.uint8).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    return x, np.asarray(d[b'labels'], dtype=np.int64)

def load_cifar10():
    folder, source = locate_cifar10()
    parts = [read_batch(folder / f'data_batch_{i}') for i in range(1, 6)]   # 5 × 10 000 de entrenamiento
    x_test, y_test = read_batch(folder / 'test_batch')                      # 10 000 de prueba
    with open(folder / 'batches.meta', 'rb') as f:
        classes = [c.decode() for c in pickle.load(f, encoding='bytes')[b'label_names']]
    print('Fuente:', source, '·', folder)
    return np.concatenate([x for x, _ in parts]), np.concatenate([y for _, y in parts]), x_test, y_test, classes

x_train_raw, y_train_raw, x_test_raw, y_test_raw, CLASSES = load_cifar10()
x_train_raw.shape, x_test_raw.shape

In [ ]:
X_train = torch.from_numpy(x_train_raw).permute(0, 3, 1, 2).to(device)    # uint8, N × 3 × 32 × 32
X_test = torch.from_numpy(x_test_raw).permute(0, 3, 1, 2).to(device)
y_train, y_test = torch.from_numpy(y_train_raw).to(device), torch.from_numpy(y_test_raw).to(device)
CIFAR = (torch.tensor([0.4914, 0.4822, 0.4465], device=device).view(1, 3, 1, 1),
         torch.tensor([0.2470, 0.2435, 0.2616], device=device).view(1, 3, 1, 1))
IMNET = (torch.tensor(ResNet18_Weights.IMAGENET1K_V1.transforms().mean, device=device).view(1, 3, 1, 1),
         torch.tensor(ResNet18_Weights.IMAGENET1K_V1.transforms().std, device=device).view(1, 3, 1, 1))

## 0. Los cuatro encoders
Cada uno viene con su preprocesamiento: la normalización con la que se entrenó y, para el de ImageNet, el tamaño (las imágenes se amplían a 96×96, porque esa red se entrenó con fotos de 224×224 y a 32×32 pierde mucho).

El **autoencoder** usa la misma ResNet-18 como encoder $g$ y un decoder de convoluciones transpuestas: $\mathbf h \in \mathbb R^{512} \to 256\times4\times4 \to \dots \to 3\times32\times32$. Lo entrenamos aquí, 10 épocas, unos 3 minutos.

In [ ]:
def resnet18_cifar():                                    # la del notebook 4
    m = resnet18(weights=None)
    m.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False); m.maxpool = nn.Identity(); m.fc = nn.Identity()
    return m

def prep_cifar(xb):
    return (xb.float() / 255 - CIFAR[0]) / CIFAR[1]

def prep_imagenet(xb):
    x = F.interpolate(xb.float() / 255, size=96, mode='bilinear', align_corners=False)
    return (x - IMNET[0]) / IMNET[1]

encoders = {}
torch.manual_seed(0)
encoders['aleatorio'] = (resnet18_cifar().to(device), prep_cifar)

g_in = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1); g_in.fc = nn.Identity()
encoders['ImageNet'] = (g_in.to(device), prep_imagenet)


CKPT_HANDLE = 'juanmartinezv4399/si7011-s04-simclr'   # modelo de Kaggle con los checkpoints largos, con y sin color
CKPT_VARIANTE = 'pyTorch/default'                      # framework/variación del modelo en Kaggle

def buscar(nombre):
    # orden: carpeta del notebook y data/, input de Kaggle, Lightning, y por último kagglehub con tu token
    for d in [Path('.'), *ROOTS]:
        hits = sorted(d.glob(f'**/{nombre}')) if d.exists() else []
        if hits:
            return hits[0]
    if kaggle_token():
        try:
            import kagglehub
            hits = sorted(Path(kagglehub.model_download(f'{CKPT_HANDLE}/{CKPT_VARIANTE}')).glob(f'**/{nombre}'))
            if hits:
                return hits[0]
        except Exception as e:
            print('kagglehub:', type(e).__name__, '· no se pudo bajar', CKPT_HANDLE)
    return None

ruta = buscar('simclr_resnet18_cifar10.pt') or buscar('simclr_g.pt')
assert ruta is not None, f'Falta el encoder SimCLR: agrega el modelo {CKPT_HANDLE} (Add Input → Models) o corre el notebook 4.'
g_sim = resnet18_cifar().to(device); g_sim.load_state_dict(torch.load(ruta, map_location=device)['g'])
encoders['SimCLR'] = (g_sim, prep_cifar)
print('SimCLR desde', ruta)

In [ ]:
AE_EPOCAS = 10
torch.manual_seed(0)
g_ae = resnet18_cifar().to(device)
d_ae = nn.Sequential(nn.Linear(512, 256 * 4 * 4), nn.ReLU(), nn.Unflatten(1, (256, 4, 4)),
                     nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(),     # → 8×8
                     nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(),       # → 16×16
                     nn.ConvTranspose2d(64, 3, 4, 2, 1), nn.Sigmoid()).to(device)               # → 3×32×32 en [0, 1]
opt = torch.optim.Adam([*g_ae.parameters(), *d_ae.parameters()], lr=1e-3)
scaler = torch.amp.GradScaler(enabled=amp)
for ep in range(AE_EPOCAS):
    t0, tot = time.time(), 0.0
    for b in torch.randperm(len(X_train), device=device).split(256):
        xb = X_train[b]
        with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
            loss = F.mse_loss(d_ae(g_ae(prep_cifar(xb))).float(), xb.float() / 255)    # ‖x − d(g(x))‖², sin etiquetas
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item() * len(b)
    print(f'autoencoder · época {ep + 1} · MSE {tot / len(X_train):.4f} · {time.time() - t0:.0f} s')
encoders['autoencoder'] = (g_ae, prep_cifar)
encoders = {k: encoders[k] for k in ['aleatorio', 'autoencoder', 'SimCLR', 'ImageNet']}

### Las features, una sola vez
Los cuatro encoders quedan **congelados**: calculamos $\mathbf h = g(\mathbf x)$ para las 60 000 imágenes y lo guardamos (512 números por imagen), como en el notebook 4 de la sesión 03. Todo lo que sigue, salvo la sección 6, trabaja sobre estas features y tarda segundos.

In [ ]:
@torch.no_grad()
def features(g, prep, X, chunk=500):
    g.eval()
    with torch.autocast(device.type, dtype=torch.float16, enabled=amp):
        return torch.cat([g(prep(X[i:i + chunk])).float() for i in range(0, len(X), chunk)])

t0 = time.time()
H = {n: (features(g, pr, X_train), features(g, pr, X_test)) for n, (g, pr) in encoders.items()}
print(f'features en {time.time() - t0:.0f} s ·', {n: tuple(h[0].shape) for n, h in H.items()})

## 1. $k$-NN: medir sin entrenar nada
El mismo $k$-NN del notebook 4: los 20 vecinos de entrenamiento más parecidos por coseno, con voto ponderado.

**Predice:** ordena los cuatro encoders antes de correr la celda.

In [ ]:
@torch.no_grad()
def knn(Htr, ytr, Hte, k=20, t=0.07):
    Htr, Hte = F.normalize(Htr, dim=1), F.normalize(Hte, dim=1)
    k = min(k, len(Htr))
    pred = []
    for i in range(0, len(Hte), 1000):
        sim, j = (Hte[i:i + 1000] @ Htr.T).topk(k, dim=1)
        votos = torch.zeros(len(j), 10, device=Hte.device).scatter_add_(1, ytr[j], (sim / t).exp())
        pred.append(votos.argmax(1))
    return (torch.cat(pred) == y_test).float().mean().item()

res_knn = {n: knn(Htr, y_train, Hte) for n, (Htr, Hte) in H.items()}
{n: round(v, 3) for n, v in res_knn.items()}

## 2. Recuperación: "muéstrame imágenes como esta"
El $k$-NN también es un buscador. Para 5 imágenes de prueba mostramos sus 6 vecinos de entrenamiento con $\mathbf h$ de SimCLR. Marco verde: misma clase; rojo: otra clase.

In [ ]:
consultas = torch.tensor([3, 21, 112, 1205, 4031], device=device)
Htr, Hte = (F.normalize(h, dim=1) for h in H['SimCLR'])
sim, vec = (Hte[consultas] @ Htr.T).topk(6, dim=1)
fig, axes = plt.subplots(5, 7, figsize=(9, 6.6))
for r, q in enumerate(consultas):
    axes[r, 0].imshow(X_test[q].permute(1, 2, 0).cpu()); axes[r, 0].set_title(CLASSES[y_test[q]], fontsize=7)
    for c in range(6):
        j = vec[r, c]; ok = y_train[j] == y_test[q]
        ax = axes[r, c + 1]; ax.imshow(X_train[j].permute(1, 2, 0).cpu())
        ax.set_title(f'{sim[r, c]:.2f}', fontsize=7)
        for s in ax.spines.values(): s.set(color='tab:green' if ok else 'tab:red', lw=3)
for ax in axes.flat: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
np.savez_compressed('recuperacion_simclr.npz', consultas=consultas.cpu().numpy(), vecinos=vec.cpu().numpy(),
                    similitud=sim.cpu().numpy())                       # figura S04-F08

Mira los errores: cuando un vecino es de otra clase, suele compartir el **fondo** o el color dominante (un barco y un avión sobre azul, un ciervo y un caballo sobre pasto). Es lo que las aumentaciones no le enseñaron a ignorar.

## 3. *Linear probe* con todas las etiquetas
Una capa lineal sobre $\mathbf h$ congelado, entrenada con las 50 000 etiquetas. Es el techo de lo que una regla lineal puede leer de cada representación.

In [ ]:
def probe(Htr, ytr, Hte, steps=300, wd=1e-4, seed=0):
    mu, sd = Htr.mean(0), Htr.std(0) + 1e-6 if len(Htr) > 1 else torch.ones_like(Htr[0])
    torch.manual_seed(seed)
    head = nn.Linear(Htr.shape[1], 10).to(Htr.device)          # ŷ = softmax(W h + b): lo único que se entrena
    opt = torch.optim.Adam(head.parameters(), lr=1e-2, weight_decay=wd)
    for _ in range(steps):
        loss = F.cross_entropy(head((Htr - mu) / sd), ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return (head((Hte - mu) / sd).argmax(1) == y_test).float().mean().item()

res_probe = {n: probe(Htr, y_train, Hte) for n, (Htr, Hte) in H.items()}
for n in H:
    print(f'{n:12s} · k-NN {res_knn[n]:.3f} · linear probe {res_probe[n]:.3f}')

Las dos lecturas suelen dar el mismo orden, pero no siempre la misma distancia entre encoders. El autoencoder gana menos con la *probe* que SimCLR: reconstruir obliga a guardar color y textura, información que está en $\mathbf h$ pero no organizada por clase. Y el de ImageNet, con un millón de etiquetas de otro dataset y fotos más grandes, sigue siendo un rival difícil.

## 4. ¿Cuántas etiquetas hacen falta?
Es la pregunta que justifica la sesión (diapositiva *Label efficiency*). Repetimos la *probe* con 1, 10, 100, 1000 y 5000 imágenes etiquetadas por clase, tres sorteos distintos para cada tamaño. Las features no cambian: solo cuántas filas ve la capa lineal.

In [ ]:
POR_CLASE, SEMILLAS = [1, 10, 100, 1000, 5000], [0, 1, 2]
por_clase_idx = [(y_train == c).nonzero().squeeze(1) for c in range(10)]

def submuestra(n, seed):
    g = torch.Generator(device='cpu').manual_seed(seed)
    return torch.cat([idx[torch.randperm(len(idx), generator=g)[:n].to(idx.device)] for idx in por_clase_idx])

eficiencia = {n: {str(k): [] for k in POR_CLASE} for n in H}
t0 = time.time()
for n, (Htr, Hte) in H.items():
    for k in POR_CLASE:
        for s in SEMILLAS:
            i = submuestra(k, s)
            eficiencia[n][str(k)].append(probe(Htr[i], y_train[i], Hte, seed=s))
print(f'{len(H) * len(POR_CLASE) * len(SEMILLAS)} probes en {time.time() - t0:.0f} s')

colores = {'aleatorio': 'gray', 'autoencoder': 'tab:blue', 'SimCLR': 'tab:green', 'ImageNet': 'tab:orange'}
plt.figure(figsize=(6.2, 3.8))
for n, r in eficiencia.items():
    m = np.array([np.mean(r[str(k)]) for k in POR_CLASE]); lo = np.array([min(r[str(k)]) for k in POR_CLASE]); hi = np.array([max(r[str(k)]) for k in POR_CLASE])
    plt.plot(POR_CLASE, m, 'o-' if n != 'ImageNet' else 'o--', color=colores[n], label=n)
    plt.fill_between(POR_CLASE, lo, hi, color=colores[n], alpha=0.15)
plt.axhline(0.1, color='k', ls=':', lw=1, label='azar')
plt.xscale('log'); plt.xticks(POR_CLASE, POR_CLASE); plt.minorticks_off(); plt.ylim(0, 1)
plt.xlabel('imágenes etiquetadas por clase'); plt.ylabel('accuracy de prueba (linear probe)'); plt.legend(fontsize=8); plt.show()
json.dump({'por_clase': POR_CLASE, 'semillas': SEMILLAS, 'resultados': eficiencia},
          open('resultados_eficiencia.json', 'w'), indent=1)                 # figura S04-F09

Lee la gráfica de izquierda a derecha. Con **una** imagen por clase, una *probe* sobre SimCLR ya está muy por encima del azar; con 100, se acerca a lo que alcanza con las 50 000. La distancia entre SimCLR y el encoder aleatorio, a la izquierda, es el valor de las 50 000 imágenes **sin etiquetar**: es lo que se gana sin pagarle a nadie por anotar.

## 5. Mirar las representaciones: t-SNE
Proyectamos a 2D las mismas 2 000 imágenes de prueba con tres encoders (PCA a 50 dimensiones y luego t-SNE). El color es la clase, que ningún encoder vio.

In [ ]:
idx2k = torch.randperm(len(X_test), generator=torch.Generator().manual_seed(0))[:2000]
proy = {}
fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, n in zip(axes, ['aleatorio', 'autoencoder', 'SimCLR']):
    Z = PCA(50, random_state=0).fit_transform(H[n][1][idx2k.to(device)].cpu().numpy())
    proy[n] = TSNE(2, perplexity=30, init='pca', random_state=0).fit_transform(Z)
    sc = ax.scatter(*proy[n].T, c=y_test[idx2k.to(device)].cpu(), cmap='tab10', s=3)
    ax.set_title(n); ax.set_xticks([]); ax.set_yticks([])
fig.legend(sc.legend_elements()[0], CLASSES, loc='lower center', ncol=10, fontsize=8, markerscale=0.8)
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()
np.savez_compressed('embeddings_test.npz', indices=idx2k.numpy(), y=y_test[idx2k.to(device)].cpu().numpy(),
                    **{n: z for n, z in proy.items()})                       # figura S04-F10

SimCLR forma grupos por clase; el aleatorio, casi ninguno. Pero cuidado (diapositiva *Looking at embeddings*): t-SNE conserva **vecinos**, no distancias. El tamaño de cada grupo y el espacio entre grupos no significan nada, y con otra semilla la figura cambia. Para comparar encoders usa los números de las secciones 1, 3 y 4; t-SNE sirve para mirar, no para medir.

## 6. Con 10 etiquetas por clase: ¿$k$-NN, *probe* o *fine-tuning*?
Es la pregunta de la diapositiva. Tenemos 100 imágenes etiquetadas y el encoder de SimCLR:

- **$k$-NN** sobre $\mathbf h$, con solo esas 100 como banco de vecinos;
- ***linear probe*** sobre $\mathbf h$ congelado;
- ***fine-tuning***: $g$ y una cabeza nueva, todo entrenable, con aumentaciones de recorte y espejo y $\eta$ menor para $g$ (sesión 03, notebook 4).

**Predice** cuál gana antes de correr.

In [ ]:
i10 = submuestra(10, 0)
Htr, Hte = H['SimCLR']
res10 = {'k-NN': knn(Htr[i10], y_train[i10], Hte, k=5), 'linear probe': probe(Htr[i10], y_train[i10], Hte)}

def aumentar(xb, pad=4):                                   # recorte con traslación y espejo, en la GPU
    flip = torch.rand(len(xb), device=xb.device) < 0.5
    xb = torch.where(flip.view(-1, 1, 1, 1), xb.flip(3), xb)
    xp = F.pad(xb, (pad,) * 4, mode='reflect'); ij = torch.randint(0, 2 * pad + 1, (len(xb), 2))
    return torch.stack([xp[k, :, i:i + 32, j:j + 32] for k, (i, j) in enumerate(ij.tolist())])

def finetune(g0, X, y, epocas=60, lr_g=1e-3, lr_head=1e-2):
    torch.manual_seed(0)
    g = copy.deepcopy(g0).train(); head = nn.Linear(512, 10).to(device)
    opt = torch.optim.AdamW([{'params': g.parameters(), 'lr': lr_g}, {'params': head.parameters(), 'lr': lr_head}],
                            weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epocas * math.ceil(len(X) / 50))
    for _ in range(epocas):
        for b in torch.randperm(len(X), device=device).split(50):
            loss = F.cross_entropy(head(g(aumentar(prep_cifar(X[b])))), y[b])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    g.eval()
    with torch.no_grad():
        return (torch.cat([head(g(prep_cifar(X_test[i:i + 1000]))) for i in range(0, len(X_test), 1000)]).argmax(1)
                == y_test).float().mean().item()

t0 = time.time()
res10['fine-tuning'] = finetune(g_sim, X_train[i10], y_train[i10])
res10['fine-tuning desde cero'] = finetune(encoders['aleatorio'][0], X_train[i10], y_train[i10])
print(f'{time.time() - t0:.0f} s ·', {n: round(v, 3) for n, v in res10.items()})

Con tan pocas etiquetas, descongelar 11 millones de parámetros arriesga sobreajustar las 100 imágenes; la *probe* y el $k$-NN no pueden dañar $\mathbf h$. Compara también las dos filas de *fine-tuning*: la misma receta, desde pesos aleatorios, muestra cuánto aporta el preentrenamiento cuando casi no hay etiquetas.

## Tu turno · 10 minutos
1. Repite la sección 4 con $k$-NN en vez de *probe* (usa `knn(Htr[i], y_train[i], Hte, k=min(20, len(i)))`). ¿Con pocas etiquetas, cuál de las dos lecturas es mejor?
2. En la sección 6, sube a 100 etiquetas por clase. ¿Cambia el ganador?
3. Busca en la sección 2 una consulta donde todos los vecinos sean de otra clase. ¿Qué tienen en común con la consulta?

**Para pensar.** El encoder de ImageNet usó más de un millón de etiquetas humanas. ¿Contra cuál de los otros encoders es justo compararlo? ¿Qué pasaría con imágenes muy distintas de las de ImageNet (diapositiva *When transfer helps less* de la sesión 03)? Lo pruebas en el integrador.

In [ ]:
# Experimento 1: eficiencia de etiquetas con k-NN

### Referencias
- Chen et al. (2020), *A Simple Framework for Contrastive Learning of Visual Representations*, ICML (evaluación con *linear probe* y con 1 % y 10 % de las etiquetas).
- Wu et al. (2018), *Unsupervised Feature Learning via Non-Parametric Instance Discrimination*, CVPR ($k$-NN ponderado).
- van der Maaten y Hinton (2008), *Visualizing Data using t-SNE*, JMLR; Wattenberg, Viégas y Johnson (2016), *How to Use t-SNE Effectively*, Distill.